In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Matryoshka Dendrogram – Entity-Aware Iterative Retrieval (Ablation)
#
# Score ibrido (step 2+):
#   beta_t          = 1 - n_retrieved / K            (dinamico, calcolato a runtime)
#   score_centroide = cos(q_orig^l, c) + beta_t * cos(q_exp^l, c)   [discesa dendrogramma]
#   S_sim(d)        = cos(q_orig, d)  + beta_t * cos(q_exp, d)      [rerank leaf]
#   score_finale(d) = S_sim(d) + BETA_IOU * IoU(seen_ents, d)       [rerank finale, step 2+]
#
# Step 1: score(d) = cos(q_orig, d)  — puro coseno in tutta la discesa
#
# Ablation su:
#   IOU_RERANK             : [None, 0.3, 0.5]  (None → BETA_IOU=0.0)
#   TOP_K_CLUSTERS_PER_DOC : [1, 2, 3, 4]      (ogni valore ha il suo indice salvato)
#   MAX_CTX_CHUNKS         : [5, 10]
#
# Path indici: SAVE_DIR/topk_<N>/
# Path run   : _ABLATION_ROOT/<suffix>/ablation_result.json
# Sommario   : _ABLATION_ROOT/ablation_summary.csv
#
# Resume-safe: run e indici già presenti vengono riutilizzati.


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 1 – Avvio Ollama
# ══════════════════════════════════════════════════════════════════════════════

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "7"
env["OLLAMA_HOST"]             = "127.0.0.1:11506"
env["OLLAMA_NUM_PARALLEL"]     = "4"
env["OLLAMA_MAX_LOADED_MODELS"] = "4"

process = subprocess.Popen(["ollama", "serve"], env=env)
print("Ollama avviato, PID:", process.pid)

device = 'cuda'
print(f'Device: {device}')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 2 – Imports
# ══════════════════════════════════════════════════════════════════════════════

import os, json, re, string, hashlib, asyncio, time, pickle, itertools as _itertools
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Set, Optional, Tuple, Union
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import hdbscan
import faiss
from gliner import GLiNER
from openai import OpenAI
from tqdm import tqdm
print('Imports OK')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 – Iperparametri
# ══════════════════════════════════════════════════════════════════════════════

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_NAME = 'musique'   # 'hotpotqa' | '2wikimultihopqa'
QA_FILE      = f'../../dataset/{DATASET_NAME}.json'
CORPUS_FILE  = f'../../dataset/{DATASET_NAME}_corpus.json'

# ── LLM (OpenAI-compatible) ───────────────────────────────────────────────────
LLM_MODEL      = 'gemma3:27b-it-qat'
API_BASE_URL   = 'http://localhost:11506/v1'
API_KEY        = 'not-needed'
MAX_NEW_TOKENS = 1000
TEMPERATURE    = 0
TOP_P          = 0.9

# ── Embedding ─────────────────────────────────────────────────────────────────
EMBEDDING_MODEL  = 'nomic-ai/nomic-embed-text-v1.5'
CACHE_DIR        = '../../models'
EMBED_BATCH_SIZE = 512

# ── Matryoshka ────────────────────────────────────────────────────────────────
MATRYOSHKA_DIMS = [64, 128, 256, 512, 768]

# ── HDBSCAN ───────────────────────────────────────────────────────────────────
CLUSTER_SELECTION_METHOD = 'leaf'
TOP_K_CLUSTERS_PER_DOC   = 3       # default; sovrascritto dall'ablation
HDBSCAN_MIN_CLUSTER_SIZE = 0       # 0 = adattivo
HDBSCAN_MIN_SAMPLES      = 10
SOFT_THRESHOLD           = 0.25
USE_UMAP                 = True
UMAP_N_COMPONENTS        = 64

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K_PER_LEVEL = [8, 16, 32, 64, 128]
MAX_CTX_CHUNKS  = 10    # default; sovrascritto dall'ablation
K_DOCS          = 10

# ── Backend indice ────────────────────────────────────────────────────────────
INDEX_TYPE           = 'numpy'
HNSW_M               = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH       = 64

# ── Reranker ──────────────────────────────────────────────────────────────────
RERANKER            = 'biencoder'
CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

# ── Score ibrido (step 2+) ────────────────────────────────────────────────────
# beta_t    = 1 - n_retrieved / K  (dinamico, calcolato a runtime)
# BETA_IOU  = peso del contributo IoU nel reranking finale
ALPHA = 0.5  # default sovrascritto dall'ablation

# ── Chunking ──────────────────────────────────────────────────────────────────
CHUNK_ENABLED   = True
CHUNK_SIZE      = 100
CHUNK_OVERLAP   = 20
CHUNK_MIN_WORDS = 10

# ── File paths ────────────────────────────────────────────────────────────────
SAVE_DIR              = f'./Indices/{DATASET_NAME}_matryoshka_index_iterative'
RESULTS_FOLDER        = f'./results/{DATASET_NAME}_iterative'
CHUNK_ENTITIES_FILE   = f'./Indices/{DATASET_NAME}_chunk_entities_chunks.pkl'
RECREATE_INDEX        = False
RECREATE_CHUNK_ENTITIES = False

# ── Benchmark ─────────────────────────────────────────────────────────────────
CHECKPOINT_EVERY = 50

# ── GLiNER ────────────────────────────────────────────────────────────────────
GLINER_MODEL        = 'urchade/gliner_medium-v2.1'
GLINER_BATCH_SIZE   = 8
GLINER_THRESHOLD    = 0.4
GLINER_DEVICE       = 'cuda'
GLINER_ENTITY_TYPES = ['person', 'location', 'organization',
                       'event', 'product', 'work of art']
GLINER_GRAPH_TYPES  = {'person', 'location', 'organization',
                       'event', 'product', 'work of art'}

SIM_MODE = "beta"   # "beta" | "query_base" | "query_espansa"

# ── Ablation ──────────────────────────────────────────────────────────────────
_ABLATION_GRID = {
    "IOU_RERANK"             : [0.5],#[1,  0.85, 0.75, 0.65, 0.5, 0.25, None],   # None → BETA_IOU=0.0
    "TOP_K_CLUSTERS_PER_DOC" : [2], #[1, 2, 3, 4],
    "MAX_CTX_CHUNKS"         : [10],#[5, 10],
    'SIM_MODE'               : ['beta', 'query_base', 'query_espansa'],  # ← nuovo
}
ABLATION_N_SAMPLES = None
_ABLATION_ROOT     = f'./results/{DATASET_NAME}_iterative/ablation_runs'

assert INDEX_TYPE in ('numpy', 'faiss_flat', 'faiss_hnsw')
assert RERANKER   in ('biencoder', 'crossencoder')

print(f'DATASET        = {DATASET_NAME}')
print(f'INDEX_TYPE     = {INDEX_TYPE}')
print(f'RERANKER       = {RERANKER}')
print(f'K_DOCS         = {K_DOCS}')
print(f'LLM_MODEL      = {LLM_MODEL}')
print(f'EMBED_MODEL    = {EMBEDDING_MODEL}')
print(f'RESULTS_FOLDER = {RESULTS_FOLDER}')
print(f'ABLATION combos= {len(list(_itertools.product(*_ABLATION_GRID.values())))}')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 4 – Caricamento dati
# ══════════════════════════════════════════════════════════════════════════════

with open(QA_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f'Campioni QA: {len(data)}')

try:
    with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
        corpus = json.load(f)
    print(f'Corpus grezzo: {len(corpus)} documenti')
except FileNotFoundError:
    print(f'[WARN] Corpus non trovato in {CORPUS_FILE}.')
    corpus = []


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 5 – Chunking e corpus
# ══════════════════════════════════════════════════════════════════════════════

def chunk_text(title: str, text: str,
               chunk_size: int = CHUNK_SIZE,
               overlap: int    = CHUNK_OVERLAP,
               min_words: int  = CHUNK_MIN_WORDS) -> List[Dict]:
    words  = text.split()
    chunks = []
    step   = max(1, chunk_size - overlap)
    for i, start in enumerate(range(0, len(words), step)):
        w = words[start: start + chunk_size]
        if len(w) < min_words:
            break
        chunks.append({'title': title, 'text': ' '.join(w),
                       'chunk_idx': i, 'parent_title': title})
    return chunks if chunks else [{'title': title, 'text': text,
                                   'chunk_idx': 0, 'parent_title': title}]


if CHUNK_ENABLED:
    documents = []
    for e in corpus:
        documents.extend(chunk_text(e.get('title', ''), e.get('text', '')))
    print(f'Chunking attivo: {len(corpus)} doc → {len(documents)} chunk '
          f'(size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})')
else:
    documents = [{'title': e.get('title', ''), 'text': e.get('text', '')}
                 for e in corpus]
    print(f'Chunking disattivo: {len(documents)} documenti')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 6 – Utilities
# ══════════════════════════════════════════════════════════════════════════════

def generate_doc_id(title: str, text: str) -> str:
    normalized = re.sub(r'\s+', '', text)
    return hashlib.md5(f"{title.strip()}{normalized}".encode('utf-8')).hexdigest()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 7 – VectorIndex
# ══════════════════════════════════════════════════════════════════════════════

class VectorIndex:
    def __init__(self, backend='numpy', hnsw_m=32,
                 hnsw_ef_construction=200, hnsw_ef_search=64):
        assert backend in ('numpy', 'faiss_flat', 'faiss_hnsw')
        self.backend = backend
        self.hnsw_m = hnsw_m
        self.hnsw_ef_construction = hnsw_ef_construction
        self.hnsw_ef_search = hnsw_ef_search
        self._vectors = None
        self._faiss   = None
        self.ntotal   = 0

    def build(self, vectors: np.ndarray) -> 'VectorIndex':
        v = vectors.astype(np.float32)
        self.ntotal = v.shape[0]
        if self.backend == 'numpy':
            self._vectors = v
        elif self.backend == 'faiss_flat':
            self._faiss = faiss.IndexFlatIP(v.shape[1])
            self._faiss.add(v)
        else:
            self._faiss = faiss.IndexHNSWFlat(v.shape[1], self.hnsw_m)
            self._faiss.hnsw.efConstruction = self.hnsw_ef_construction
            self._faiss.hnsw.efSearch       = self.hnsw_ef_search
            self._faiss.add(v)
        return self

    def search(self, query: np.ndarray, k: int) -> List[int]:
        k = min(k, self.ntotal)
        q = query.ravel().astype(np.float32)
        if self.backend == 'numpy':
            return np.argsort((self._vectors @ q).ravel())[::-1][:k].tolist()
        _, ids = self._faiss.search(q.reshape(1, -1), k)
        return ids[0].tolist()

    def save(self, path: str):
        if self.backend == 'numpy':
            np.save(path + '.npy', self._vectors)
        else:
            faiss.write_index(self._faiss, path + '.faiss')

    def load(self, path: str) -> 'VectorIndex':
        if self.backend == 'numpy':
            self._vectors = np.load(path + '.npy')
            self.ntotal   = len(self._vectors)
        else:
            self._faiss = faiss.read_index(
                path + '.faiss' if self.backend == 'faiss_flat' else path + '.faiss'
            )
            self.ntotal = self._faiss.ntotal
        return self
    
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 8 – LevelIndex dataclass
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class LevelIndex:
    dim:            int
    n_clusters:     int
    centroids:      np.ndarray
    soft_membership: np.ndarray
    doc_clusters:   List[List[Tuple[int, float]]]
    index:          VectorIndex
    children:       Dict[int, Set[int]] = field(default_factory=dict)


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 9 – EntityExtractor + build_chunk_entities
# ══════════════════════════════════════════════════════════════════════════════

class EntityExtractor:
    def __init__(self, model_name=GLINER_MODEL, entity_types=None,
                 batch_size=GLINER_BATCH_SIZE, threshold=GLINER_THRESHOLD,
                 device=GLINER_DEVICE):
        self.model_name   = model_name
        self.entity_types = entity_types or GLINER_ENTITY_TYPES
        self.batch_size   = batch_size
        self.threshold    = threshold
        self.device       = device
        self._model       = None

    def _load(self):
        if self._model is None:
            print(f'  Carico GLiNER {self.model_name} ...')
            self._model = GLiNER.from_pretrained(self.model_name)
            self._model.to(self.device)
            print('  GLiNER caricato.')

    @staticmethod
    def _normalize(text: str) -> str:
        return re.sub(r'\s+', ' ', text.lower().strip())

    def extract_batch(self, texts: List[str],
                      graph_types: Set[str] = None) -> List[Set[str]]:
        self._load()
        _graph_types = graph_types if graph_types is not None else GLINER_GRAPH_TYPES
        all_entities = []
        import torch
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                batch = texts[i: i + self.batch_size]
                preds_list = self._model.batch_predict_entities(
                    batch, self.entity_types, threshold=self.threshold
                )
                for preds in preds_list:
                    ents = {
                        self._normalize(p['text'])
                        for p in preds
                        if p['score'] >= self.threshold
                        and len(p['text'].strip()) > 1
                        and p['label'] in _graph_types
                    }
                    all_entities.append(ents)
                done = min(i + self.batch_size, len(texts))
                if done % 1000 == 0 or done == len(texts):
                    print(f'    [NER] {done}/{len(texts)} chunk processati')
        return all_entities

    def extract_one(self, text: str, graph_types: Set[str] = None) -> Set[str]:
        self._load()
        _graph_types = graph_types if graph_types is not None else GLINER_GRAPH_TYPES
        import torch
        with torch.no_grad():
            preds = self._model.predict_entities(
                text, self.entity_types, threshold=self.threshold
            )
        return {
            self._normalize(p['text'])
            for p in preds
            if p['score'] >= self.threshold
            and len(p['text'].strip()) > 1
            and p['label'] in _graph_types
        }


def build_chunk_entities(
    documents:  List[Dict],
    extractor:  EntityExtractor,
    save_path:  str = CHUNK_ENTITIES_FILE,
) -> Dict[int, frozenset]:
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    if not RECREATE_CHUNK_ENTITIES and os.path.isfile(save_path):
        with open(save_path, 'rb') as fh:
            ents = pickle.load(fh)
        print(f'[OK] Chunk entities caricate da {save_path} ({len(ents)} chunk)')
        return ents
    print(f'[1/2] EntityExtractor — modello: {extractor.model_name}')
    print(f'[2/2] Estraggo entità da {len(documents)} chunk ...')
    t0 = time.perf_counter()
    texts     = [f"{d.get('title', '')}\n{d.get('text', '')}" for d in documents]
    ents_list = extractor.extract_batch(texts)
    chunk_entities = {i: frozenset(e) for i, e in enumerate(ents_list)}
    with open(save_path, 'wb') as fh:
        pickle.dump(chunk_entities, fh)
    print(f'[OK] Chunk entities salvate in {save_path}')
    t_total = time.perf_counter() - t0
    
    print(t_total)

    return chunk_entities


# Matryoshka

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 10 – MatryoshkaDendrogramRetriever
# ══════════════════════════════════════════════════════════════════════════════

class MatryoshkaDendrogramRetriever:
    """
    Retriever gerarchico a dendrogramma basato su Matryoshka embeddings.

    Score ibrido (step 2+):
      beta_t             = 1 - n_retrieved / K
      score_centroide(c) = cos(q_orig^l, c) + beta_t * cos(q_exp^l, c)
      S_sim(d)           = cos(q_orig, d)   + beta_t * cos(q_exp, d)
      score_finale(d)    = S_sim(d) + BETA_IOU * IoU(seen_ents, d)

    Step 1: beta_t=0 → score = cos(q_orig, d) ovunque.
    """

    def __init__(
        self,
        embedding_model          = EMBEDDING_MODEL,
        cache_dir                = CACHE_DIR,
        matryoshka_dims          = None,
        hdbscan_min_cluster_size = HDBSCAN_MIN_CLUSTER_SIZE,
        hdbscan_min_samples      = HDBSCAN_MIN_SAMPLES,
        cluster_selection_method = CLUSTER_SELECTION_METHOD,
        soft_threshold           = SOFT_THRESHOLD,
        top_k_clusters_per_doc   = TOP_K_CLUSTERS_PER_DOC,
        use_umap                 = USE_UMAP,
        umap_n_components        = UMAP_N_COMPONENTS,
        top_k_per_level          = None,
        embed_batch_size         = EMBED_BATCH_SIZE,
        index_type               = INDEX_TYPE,
        hnsw_m                   = HNSW_M,
        hnsw_ef_construction     = HNSW_EF_CONSTRUCTION,
        hnsw_ef_search           = HNSW_EF_SEARCH,
        reranker                 = RERANKER,
        cross_encoder_model      = CROSS_ENCODER_MODEL,
        random_seed: int = 42,           # ← aggiungi questo

    ):
        self.embedding_model          = embedding_model
        self.cache_dir                = cache_dir
        self._st_model                = None
        self.dims                     = matryoshka_dims or MATRYOSHKA_DIMS
        self.hdbscan_min_cluster_size = hdbscan_min_cluster_size
        self.hdbscan_min_samples      = hdbscan_min_samples
        self.cluster_selection_method = cluster_selection_method
        self.soft_threshold           = soft_threshold
        self.top_k_clusters_per_doc   = top_k_clusters_per_doc
        self.use_umap                 = use_umap
        self.umap_n_components        = umap_n_components
        self.embed_batch_size         = embed_batch_size
        self.index_type               = index_type
        self.hnsw_m                   = hnsw_m
        self.hnsw_ef_construction     = hnsw_ef_construction
        self.hnsw_ef_search           = hnsw_ef_search
        self.reranker                 = reranker
        self.cross_encoder_model_name = cross_encoder_model
        self._cross_encoder           = None
        self._rng = np.random.default_rng(seed=random_seed)  # ← aggiungi questo
        self.random_seed = random_seed   # ← salva per config.json


        base = top_k_per_level or TOP_K_PER_LEVEL
        self.top_k_per_level = list(base)
        while len(self.top_k_per_level) < len(self.dims):
            self.top_k_per_level.append(self.top_k_per_level[-1] * 2)

        self.documents:      list                  = []
        self.doc_embeddings: Optional[np.ndarray]  = None
        self.levels:         List[LevelIndex]      = []
        self.leaf_index:     Optional[VectorIndex] = None

        print(f'Retriever | idx={index_type} | reranker={reranker} | '
              f'overlap=top{top_k_clusters_per_doc} | '
              f"min_cl={'adattivo' if hdbscan_min_cluster_size == 0 else hdbscan_min_cluster_size}")

    # ── Embedding ─────────────────────────────────────────────────────────────
    def _get_st_model(self):
        if self._st_model is None:
            from sentence_transformers import SentenceTransformer
            print(f'  Carico SentenceTransformer {self.embedding_model!r} ...')
            self._st_model = SentenceTransformer(
                self.embedding_model,
                cache_folder=self.cache_dir,
                trust_remote_code=True,
            )
            print('  SentenceTransformer caricato.')
        return self._st_model

    async def _embed_batch(self, texts: List[str]) -> np.ndarray:
        import asyncio
        loop  = asyncio.get_event_loop()
        model = self._get_st_model()

        def _encode():
            embs  = model.encode(
                texts, batch_size=self.embed_batch_size,
                show_progress_bar=False, convert_to_numpy=True,
            ).astype(np.float32)
            norms = np.linalg.norm(embs, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            return embs / norms

        return await loop.run_in_executor(None, _encode)

    @staticmethod
    def _truncnorm(embeddings: np.ndarray, dim: int) -> np.ndarray:
        mean  = embeddings.mean(axis=1, keepdims=True)
        std   = embeddings.std(axis=1,  keepdims=True) + 1e-5
        e     = ((embeddings - mean) / std)[:, :dim].astype(np.float32)
        norms = np.linalg.norm(e, axis=1, keepdims=True)
        return e / np.where(norms < 1e-9, 1e-9, norms)

    def _make_index(self) -> VectorIndex:
        return VectorIndex(
            backend              = self.index_type,
            hnsw_m               = self.hnsw_m,
            hnsw_ef_construction = self.hnsw_ef_construction,
            hnsw_ef_search       = self.hnsw_ef_search,
        )

    def _adaptive_min_cluster(self, n_docs: int, dim: int) -> int:
        if self.hdbscan_min_cluster_size > 0:
            return self.hdbscan_min_cluster_size
        max_dim = max(self.dims)
        ratio   = dim / max_dim
        return max(3, int(n_docs / (50 + 450 * ratio)))

    # ── Build livello ─────────────────────────────────────────────────────────
    def _build_level(self, embs_norm: np.ndarray, dim: int) -> LevelIndex:
        n_docs = embs_norm.shape[0]
        min_cl = self._adaptive_min_cluster(n_docs, dim)
        data   = embs_norm

        if self.use_umap and n_docs > 500:
            from umap import UMAP
            n_comp = min(self.umap_n_components, dim, n_docs - 2)
            print(f'      UMAP {dim}d → {n_comp}d ...')
            data = UMAP(
                n_components=n_comp, metric='cosine',
                random_state=42, n_jobs=-1,
            ).fit_transform(embs_norm).astype(np.float32)

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size         = min_cl,
            min_samples              = self.hdbscan_min_samples,
            metric                   = 'euclidean',
            cluster_selection_method = self.cluster_selection_method,
            prediction_data          = True,
        )
        clusterer.fit(data)
        labels = clusterer.labels_

        n_cl = int(labels.max()) + 1
        if n_cl <= 0:
            print(f'      [WARN] 0 cluster a dim={dim}: cluster unico artificiale.')
            labels = np.zeros(n_docs, dtype=int)
            n_cl   = 1

        noise_mask = labels == -1
        n_noise    = int(noise_mask.sum())

        centroids = np.zeros((n_cl, dim), dtype=np.float32)
        for c in range(n_cl):
            mask = (labels == c)
            if mask.any():
                centroids[c] = embs_norm[mask].mean(axis=0)
        c_norms    = np.linalg.norm(centroids, axis=1, keepdims=True)
        centroids /= np.where(c_norms < 1e-9, 1e-9, c_norms)

#         k_ov = min(self.top_k_clusters_per_doc, n_cl)
#         sims = embs_norm @ centroids.T

#         probs = np.maximum(sims, 0.0)
#         row_sums = probs.sum(axis=1, keepdims=True)
#         row_sums[row_sums < 1e-9] = 1.0
#         probs = probs / row_sums

#         # NON creare rng locale — usa self._rng creato nel __init__
#         soft = np.zeros((n_docs, n_cl), dtype=np.float32)
#         for i in range(n_docs):
#             sampled         = self._rng.choice(n_cl, size=k_ov, replace=True, p=probs[i])
#             unique_clusters = np.unique(sampled)
#             vals            = probs[i, unique_clusters]
#             s               = vals.sum()
#             soft[i, unique_clusters] = vals / s if s > 1e-9 else np.ones(len(unique_clusters)) / len(unique_clusters)
    
    
    
        # CODICE ATTUALE — top-k deterministico
        k_ov = min(self.top_k_clusters_per_doc, n_cl)
        sims = embs_norm @ centroids.T
        soft = np.zeros((n_docs, n_cl), dtype=np.float32)
        for i in range(n_docs):
            top_k_idx          = np.argsort(sims[i])[::-1][:k_ov]
            vals               = np.maximum(sims[i, top_k_idx], 0.0)
            s                  = vals.sum()
            soft[i, top_k_idx] = vals / s if s > 1e-9 else np.ones(k_ov) / k_ov



        doc_clusters = []
        for i in range(n_docs):
            row = [(c, float(soft[i, c]))
                   for c in range(n_cl) if soft[i, c] >= self.soft_threshold]
            if not row:
                best = int(np.argmax(soft[i]))
                row  = [(best, 1.0)]
            row.sort(key=lambda x: x[1], reverse=True)
            doc_clusters.append(row)

        avg_per_doc = float(np.mean([len(dc) for dc in doc_clusters]))
        idx = self._make_index().build(centroids)
        print(f'    dim={dim:4d}: {n_cl:4d} cluster | noise={n_noise} | '
              f'min_cl={min_cl} | avg {avg_per_doc:.2f} cl/doc | {self.index_type}')
        return LevelIndex(dim=dim, n_clusters=n_cl, centroids=centroids,
                          soft_membership=soft, doc_clusters=doc_clusters, index=idx)

    def _build_dendrogram_links(self):
        for li in range(len(self.levels) - 1):
            lc, ln = self.levels[li], self.levels[li + 1]
            ch = {c: set() for c in range(lc.n_clusters)}
            for di in range(len(self.documents)):
                for cc in {c for c, _ in lc.doc_clusters[di]}:
                    ch[cc] |= {c for c, _ in ln.doc_clusters[di]}
            lc.children = ch


    # ── build_kg ──────────────────────────────────────────────────────────────
    async def build_kg(self, documents: List[Dict]) -> None:
        self.documents = documents
        n  = len(documents)
        t0 = time.perf_counter()

        # ── [1/4] Embedding ──────────────────────────────────────────────────
        print(f'[1/4] Embedding {n} doc ({self.embedding_model}) ...')
        t_emb_start = time.perf_counter()
        self.doc_embeddings = await self._embed_batch(
            [f"{d['title']}\n{d['text']}" for d in documents]
        )
        t_emb = time.perf_counter() - t_emb_start
        print(f'  Shape: {self.doc_embeddings.shape}  ({t_emb:.1f}s)')

        # ── [2/4] Clustering ─────────────────────────────────────────────────
        print('[2/4] HDBSCAN + overlap clustering per ogni livello ...')
        t_cluster_start = time.perf_counter()
        self.levels = []
        t_per_dim = {}
        for dim in self.dims:
            print(f'  -- dim={dim} --')
            t_dim_start = time.perf_counter()
            self.levels.append(
                self._build_level(self._truncnorm(self.doc_embeddings, dim), dim)
            )
            t_per_dim[dim] = time.perf_counter() - t_dim_start
        t_cluster = time.perf_counter() - t_cluster_start
        print(f'  Clustering totale: {t_cluster:.1f}s')

        # ── [3/4] Dendrogram links ───────────────────────────────────────────
        print('[3/4] Dendrogram links parent→children ...')
        t_dendro_start = time.perf_counter()
        self._build_dendrogram_links()
        t_dendro = time.perf_counter() - t_dendro_start

        # ── [4/4] Indice foglie ──────────────────────────────────────────────
        print(f'[4/4] Indice foglie (dim={self.dims[-1]}) ...')
        t_leaf_start = time.perf_counter()
        self.leaf_index = self._make_index().build(
            self._truncnorm(self.doc_embeddings, self.dims[-1])
        )
        t_leaf = time.perf_counter() - t_leaf_start

        t_total = time.perf_counter() - t0
        print(f'[OK] build_kg completato in {t_total:.1f}s')
        self._summary()

        # ── Salvataggio tempi su file ────────────────────────────────────────
        timing_path = Path(SAVE_DIR) / f"build_kg_timings_{self.top_k_clusters_per_doc}.txt"
        timing_path.parent.mkdir(parents=True, exist_ok=True)
        with open(timing_path, 'w', encoding='utf-8') as f:
            f.write(f"build_kg — {DATASET_NAME} — {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"{'='*60}\n")
            f.write(f"Documenti/chunk    : {n}\n")
            f.write(f"Modello embedding  : {self.embedding_model}\n")
            f.write(f"{'─'*60}\n")
            f.write(f"[1/4] Embedding                : {t_emb:.3f}s\n")
            f.write(f"[2/4] Clustering (totale)      : {t_cluster:.3f}s\n")
            for dim, t_d in t_per_dim.items():
                f.write(f"       dim={dim:<4d}              : {t_d:.3f}s\n")
            f.write(f"[3/4] Dendrogram links         : {t_dendro:.3f}s\n")
            f.write(f"[4/4] Indice foglie            : {t_leaf:.3f}s\n")
            f.write(f"{'─'*60}\n")
            f.write(f"TOTALE                         : {t_total:.3f}s\n")
        print(f'[OK] Tempi salvati in {timing_path}')
    

    # ── Summary ───────────────────────────────────────────────────────────────
    def _summary(self):
        print(f'-- Dendrogramma (idx={self.index_type} | reranker={self.reranker} | '
              f'overlap=top{self.top_k_clusters_per_doc}) ' + '-' * 20)
        for li, lvl in enumerate(self.levels):
            labels      = np.array([dc[0][0] for dc in lvl.doc_clusters])
            sizes       = np.bincount(labels, minlength=lvl.n_clusters)
            avg_cl      = float(np.mean([len(dc) for dc in lvl.doc_clusters]))
            min_cl_used = self._adaptive_min_cluster(len(self.documents), lvl.dim)
            print(f'  L{li} dim={lvl.dim:4d} | {lvl.n_clusters:4d} cl | '
                  f'avg {avg_cl:.2f} cl/doc | '
                  f'sz min={sizes.min():4d} med={int(np.median(sizes)):4d} '
                  f'max={sizes.max():4d} | min_cl_used={min_cl_used}')
        print(f'  Leaf index: {self.leaf_index.ntotal} doc @ dim={self.dims[-1]}')
        print('-' * 60)

    def avg_clusters_per_level(self) -> Dict[str, float]:
        """
        Ritorna la media dei cluster di appartenenza per ogni livello.
        Chiavi: 'L0_dim64', 'L1_dim128', ... (usato in _build_row e summary).
        """
        return {
            f'L{li}_dim{lvl.dim}': round(
                float(np.mean([len(dc) for dc in lvl.doc_clusters])), 4
            )
            for li, lvl in enumerate(self.levels)
        }
    
    
    def compute_sim_score(self, v_orig: np.ndarray, v_exp: np.ndarray,
                          targets: np.ndarray, beta_t: float) -> np.ndarray:
        """
        Calcola lo score di similarità in base a SIM_MODE.
        v_orig, v_exp : vettori query (1D, già normalizzati)
        targets       : matrice (N, dim) dei target (centroidi o doc embeddings)
        beta_t        : peso della query espansa (usato solo in modalità 'beta')
        """
        if SIM_MODE == "query_base":
            return targets @ v_orig
        elif SIM_MODE == "query_espansa":
            return targets @ v_exp
        else:  # "beta"
            return targets @ v_orig + beta_t * (targets @ v_exp)

    # ══════════════════════════════════════════════════════════════════════════
    # ENTITY-AWARE ITERATIVE RETRIEVE
    # ══════════════════════════════════════════════════════════════════════════

    async def iterative_retrieve(
        self,
        query:      str,
        k:          int                  = K_DOCS,
        dim:        Optional[int]        = None,
        extractor:  'EntityExtractor'    = None,
        chunk_ents: Optional[Dict]       = None,
    ) -> Dict:
        """
        Recupero iterativo entity-aware di k documenti.

        Step 1 (prima iterazione, beta_t=0):
          score(d) = cos(q_orig, d)

        Step 2+ (beta_t = 1 - total_retrieved / k):
          score_centroide(c) = cos(q_orig^l, c) + beta_t * cos(q_exp^l, c)
          S_sim(d)           = cos(q_orig, d)   + beta_t * cos(q_exp, d)
          score_finale(d)    = S_sim(d) + BETA_IOU * IoU(seen_ents, d)

          q_orig e q_exp proiettati con _truncnorm al dim del livello l.
          seen_ents è cumulativo (tutte le entità dei chunk recuperati finora).

        n_to_retrieve per iterazione:
          - iter 1 : |entità nella query originale|
          - iter 2+: |entità NUOVE dai chunk dell'iter precedente|
        """
        _extractor  = extractor  if extractor  is not None else globals().get('entity_extractor')
        _chunk_ents = chunk_ents if chunk_ents is not None else globals().get('chunk_entities', {})
        leaf_dim    = dim or self.dims[-1]

        retrieved_docs  = []
        retrieved_idx   = set()
        seen_entities   = set()   # cumulative
        timings         = []
        t_total_start   = time.perf_counter()
        q_orig_emb: Optional[np.ndarray] = None

        if self.reranker == 'crossencoder' and self._cross_encoder is None:
            from sentence_transformers import CrossEncoder
            self._cross_encoder = CrossEncoder(self.cross_encoder_model_name)

        # ── Entità query originale ─────────────────────────────────────────────
        t_ent = time.perf_counter()
        query_ents = (frozenset(_extractor.extract_one(query))
                      if _extractor is not None else frozenset())
        ent_lookup_ms_step1 = (time.perf_counter() - t_ent) * 1000

        n_to_retrieve = max(1, len(query_ents))
        seen_entities |= query_ents

        iteration       = 0
        total_retrieved = 0

        while total_retrieved < k:
            iteration    += 1
            t_step_start  = time.perf_counter()

            # ── Contesto: query + chunk già recuperati ─────────────────────────
            context_parts  = [query] + [
                f"{d.get('title', '')}\n{d.get('text', '')}" for d in retrieved_docs
            ]
            context_text   = '\n\n'.join(str(p) for p in context_parts)
            context_tokens = len(context_text.split())

            # ── Embedding del contesto corrente ───────────────────────────────
            t_embed     = time.perf_counter()
            ctx_emb_exp = await self._embed_batch([context_text])
            embed_ms    = (time.perf_counter() - t_embed) * 1000

            # Al primo step ctx_emb_exp == embed(query originale): cachato gratis
            if iteration == 1:
                q_orig_emb = ctx_emb_exp.copy()

            # beta_t = 0 al primo step → ogni score si riduce a cos(q_orig, .)
            beta_t = 1.0 - total_retrieved / k

            # ── Funnel gerarchico ──────────────────────────────────────────────
            t_search      = time.perf_counter()
            candidate_set = set(range(len(self.documents)))

            for li, level in enumerate(self.levels):
                top_k_cl     = self.top_k_per_level[li]
                q_orig_level = self._truncnorm(q_orig_emb,  level.dim)[0]
                q_exp_level  = self._truncnorm(ctx_emb_exp, level.dim)[0]
                # beta_t=0 al primo step → cl_scores == cos(q_orig, c)
                cl_scores = self.compute_sim_score(q_orig_level, q_exp_level, level.centroids, beta_t)
                top_cl       = np.argsort(cl_scores)[::-1][:top_k_cl]
                top_cl_set   = set(top_cl.tolist())

                new_candidates = set()
                for di in candidate_set:
                    if {c for c, _ in level.doc_clusters[di]} & top_cl_set:
                        new_candidates.add(di)

                min_pool = max(n_to_retrieve * 4, self.top_k_per_level[-1])
                if len(new_candidates) < min_pool:
                    # fallback: al primo step usa q_orig, altrimenti q_exp
                    leaf_fallback = self._truncnorm(
                        q_orig_emb if iteration == 1 else ctx_emb_exp, leaf_dim
                    )[0]
                    extra = self.leaf_index.search(leaf_fallback, k=min_pool)
                    new_candidates |= set(extra)

                candidate_set = new_candidates

            # ── Rank leaf: S_sim(d) = cos(q_orig,d) + beta_t*cos(q_exp,d) ─────
            q_orig_leaf = self._truncnorm(q_orig_emb,  leaf_dim)[0]
            q_exp_leaf  = self._truncnorm(ctx_emb_exp, leaf_dim)[0]

            if candidate_set:
                cand_list = [i for i in candidate_set if i not in retrieved_idx]
                if cand_list:
                    cand_arr = np.array(cand_list, dtype=np.int64)
                    doc_embs = self._truncnorm(self.doc_embeddings[cand_arr], leaf_dim)
                    # beta_t=0 → S_sim == cos(q_orig, d) al primo step
                    s_sim = self.compute_sim_score(q_orig_leaf, q_exp_leaf, doc_embs, beta_t)

                    order    = np.argsort(s_sim)[::-1]
                    sorted_candidates = cand_arr[order].tolist()
                    s_sim_dict = {cand_arr[j]: float(s_sim[j])
                                 for j in range(len(cand_arr))}
                else:
                    sorted_candidates = []
                    s_sim_dict         = {}
            else:
                sorted_candidates = []
                s_sim_dict         = {}

            search_ms = (time.perf_counter() - t_search) * 1000

            # ── Rerank pool ───────────────────────────────────────────────────
            t_pick = time.perf_counter()
            pool   = sorted_candidates[:self.top_k_per_level[-1]]

            if pool and self.reranker == 'crossencoder' and self._cross_encoder is not None:
                pairs     = [(context_text, self.documents[i]['text']) for i in pool]
                ce_scores = self._cross_encoder.predict(pairs)
                pool      = [pool[j] for j in np.argsort(ce_scores)[::-1]]

            # ── Score finale: S_sim + BETA_IOU * IoU (solo step 2+) ───────────
            # Step 1: pool già ordinato per S_sim = cos(q_orig, d) → nessuna modifica
            # Step 2+: score_finale(d) = S_sim(d) + BETA_IOU * IoU(seen_entities, d)
            if iteration > 1 and pool:
                final_scored = []
                for doc_idx in pool:
                    doc_ents  = _chunk_ents.get(doc_idx, frozenset())
                    union     = doc_ents | seen_entities
                    iou       = len(doc_ents & seen_entities) / len(union) if union else 0.0
                    s_sim_val = s_sim_dict.get(doc_idx, 0.0)
                    final_scored.append((doc_idx, ALPHA * s_sim_val + (1 - ALPHA) * iou))
                final_scored.sort(key=lambda x: x[1], reverse=True)
                pool = [idx for idx, _ in final_scored]

            newly_picked = pool[:n_to_retrieve]
            pick_ms      = (time.perf_counter() - t_pick) * 1000

            if not newly_picked:
                print(f'[WARN] iter {iteration}: nessun chunk nuovo disponibile.')
                break

            # ── Appendi in posizione stabile ───────────────────────────────────
            for idx in newly_picked:
                doc        = dict(self.documents[idx])
                doc['id']  = generate_doc_id(doc['title'], doc['text'])
                retrieved_docs.append(doc)
                retrieved_idx.add(idx)
            total_retrieved += len(newly_picked)

            step_ms = (time.perf_counter() - t_step_start) * 1000

            # ── Entità nuove dai chunk appena recuperati ───────────────────────
            t_ent_lookup = time.perf_counter()
            new_ents = set()
            for idx in newly_picked:
                new_ents |= (_chunk_ents.get(idx, frozenset()) - seen_entities)
            ent_lookup_ms = (time.perf_counter() - t_ent_lookup) * 1000

            timings.append({
                'iteration'        : iteration,
                'n_retrieved_this' : len(newly_picked),
                'total_retrieved'  : total_retrieved,
                'n_entities_used'  : n_to_retrieve,
                'context_tokens'   : context_tokens,
                'embed_ms'         : round(embed_ms,       2),
                'search_ms'        : round(search_ms,      2),
                'pick_ms'          : round(pick_ms,        2),
                'step_ms'          : round(step_ms,        2),
                'ent_lookup_ms'    : round(ent_lookup_ms,  2),
                'beta_t'           : round(beta_t,         4),
                'entities_used'    : (sorted(query_ents) if iteration == 1
                                      else sorted(seen_entities - query_ents)),
                'new_entities'     : sorted(new_ents),
                'doc_titles'       : [self.documents[i]['title'] for i in newly_picked],
            })

            # ── Prepara prossima iterazione ────────────────────────────────────
            seen_entities |= new_ents   # aggiorna entità cumulative per IoU

            # n_to_retrieve per il prossimo step = entità nuove trovate ora
            n_to_retrieve = max(1, len(new_ents))

        total_ms = (time.perf_counter() - t_total_start) * 1000
        return {
            'docs'     : retrieved_docs,
            'timings'  : timings,
            'total_ms' : round(total_ms, 2),
        }


print('MatryoshkaDendrogramRetriever OK')


# Save retriever

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 11 – Save / Load retriever
# ══════════════════════════════════════════════════════════════════════════════

def save_retriever(ret: MatryoshkaDendrogramRetriever, path: str) -> None:
    os.makedirs(path, exist_ok=True)
    with open(f'{path}/documents.json', 'w', encoding='utf-8') as f:
        json.dump(ret.documents, f, ensure_ascii=False)
    np.save(f'{path}/doc_embeddings.npy', ret.doc_embeddings)
    cfg = {
        'dims'                   : ret.dims,
        'index_type'             : ret.index_type,
        'top_k_clusters_per_doc' : ret.top_k_clusters_per_doc,
        'random_seed'            : ret.random_seed,   # ← aggiungi questo
    }
    with open(f'{path}/config.json', 'w') as f:
        json.dump(cfg, f)
    for li, lvl in enumerate(ret.levels):
        np.save(f'{path}/centroids_L{li}.npy',      lvl.centroids)
        np.save(f'{path}/soft_L{li}.npy',           lvl.soft_membership)
        with open(f'{path}/doc_clusters_L{li}.pkl', 'wb') as f:
            pickle.dump(lvl.doc_clusters, f)
        ch_serializable = {k: list(v) for k, v in lvl.children.items()}
        with open(f'{path}/children_L{li}.json', 'w') as f:
            json.dump(ch_serializable, f)
        lvl.index.save(f'{path}/idx_L{li}')
    ret.leaf_index.save(f'{path}/idx_leaf')
    print(f'Retriever salvato in {path}/')


def load_retriever(path: str) -> MatryoshkaDendrogramRetriever:
    with open(f'{path}/documents.json', 'r', encoding='utf-8') as f:
        docs = json.load(f)
    with open(f'{path}/config.json', 'r') as f:
        cfg_saved = json.load(f)
    saved_dims       = cfg_saved['dims']
    saved_index_type = cfg_saved['index_type']
    saved_seed = cfg_saved.get('random_seed', 42)   # ← aggiungi questo
    saved_topk       = cfg_saved.get('top_k_clusters_per_doc', TOP_K_CLUSTERS_PER_DOC)
    ret = MatryoshkaDendrogramRetriever(
        matryoshka_dims          = saved_dims,
        index_type               = saved_index_type,
        top_k_clusters_per_doc   = saved_topk,
        hdbscan_min_cluster_size = HDBSCAN_MIN_CLUSTER_SIZE,
        hdbscan_min_samples      = HDBSCAN_MIN_SAMPLES,
        cluster_selection_method = CLUSTER_SELECTION_METHOD,
        soft_threshold           = SOFT_THRESHOLD,
        use_umap                 = USE_UMAP,
        umap_n_components        = UMAP_N_COMPONENTS,
        top_k_per_level          = TOP_K_PER_LEVEL,
        embed_batch_size         = EMBED_BATCH_SIZE,
        random_seed = saved_seed,        # ← aggiungi questo

    )
    ret.documents      = docs
    ret.doc_embeddings = np.load(f'{path}/doc_embeddings.npy')
    for li, dim in enumerate(saved_dims):
        cents = np.load(f'{path}/centroids_L{li}.npy')
        soft  = np.load(f'{path}/soft_L{li}.npy')
        with open(f'{path}/doc_clusters_L{li}.pkl', 'rb') as f:
            dc = pickle.load(f)
        with open(f'{path}/children_L{li}.json', 'r') as f:
            ch_raw = json.load(f)
        ch  = {int(k): set(v) for k, v in ch_raw.items()}
        idx = VectorIndex(
            backend=saved_index_type, hnsw_m=HNSW_M,
            hnsw_ef_construction=HNSW_EF_CONSTRUCTION,
            hnsw_ef_search=HNSW_EF_SEARCH,
        ).load(f'{path}/idx_L{li}')
        ret.levels.append(LevelIndex(
            dim=dim, n_clusters=cents.shape[0],
            centroids=cents, soft_membership=soft,
            doc_clusters=dc, index=idx, children=ch,
        ))
    ret.leaf_index = ret._make_index().load(f'{path}/idx_leaf')
    print(f'Retriever caricato da {path}/')
    ret._summary()
    return ret


def retriever_dir_for_topk(top_k: int) -> str:
    return os.path.join(SAVE_DIR, f'topk_{top_k}')


def index_exists_for_topk(top_k: int) -> bool:
    path = retriever_dir_for_topk(top_k)
    return (os.path.isfile(f'{path}/config.json')
            and os.path.isfile(f'{path}/documents.json'))


print('save_retriever / load_retriever OK')



# Entity Extractor

In [ ]:
entity_extractor = EntityExtractor(
    model_name   = GLINER_MODEL,
    entity_types = GLINER_ENTITY_TYPES,
    batch_size   = GLINER_BATCH_SIZE,
    threshold    = GLINER_THRESHOLD,
    device       = GLINER_DEVICE,
)
chunk_entities: Dict[int, frozenset] = build_chunk_entities(documents, entity_extractor)
print('EntityExtractor e chunk_entities pronti.')

# Setup llm

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 12 – LLM client
# ══════════════════════════════════════════════════════════════════════════════

from openai import OpenAI as _OpenAI

_llm_client = _OpenAI(base_url=API_BASE_URL, api_key=API_KEY)


async def call_llm(prompt: str, system_prompt: str = '', model: str = LLM_MODEL) -> str:
    import asyncio
    loop = asyncio.get_event_loop()

    def _sync_call():
        sys_msg = system_prompt or (
            'You are a helpful assistant. Answer based on the context provided.'
        )
        response = _llm_client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': sys_msg},
                {'role': 'user',   'content': prompt},
            ],
            max_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )
        return response.choices[0].message.content.strip()

    return await loop.run_in_executor(None, _sync_call)


print('call_llm OK')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 13 – Metriche e prompts
# ══════════════════════════════════════════════════════════════════════════════

def normalize_answer(s: str) -> str:
    def remove_articles(t): return re.sub(r'\b(a|an|the)\b', ' ', t)
    def white_space_fix(t): return ' '.join(t.split())
    def remove_punc(t):
        exc = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
        return t.translate(exc)
    return white_space_fix(remove_articles(remove_punc(s.lower())))

def compute_em(prediction: str, ground_truth: str) -> int:
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def compute_f1(prediction: str, ground_truth: str) -> float:
    pt = normalize_answer(prediction).split()
    gt = normalize_answer(ground_truth).split()
    common   = Counter(pt) & Counter(gt)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pt)
    recall    = num_same / len(gt)
    return (2 * precision * recall) / (precision + recall)

def extract_answer(llm_out: str) -> str:
    m = re.search(r'(?i)Answer[:.] *(.*)', llm_out, re.DOTALL)
    return m.group(1).strip() if m else llm_out.strip()


SYSTEM_PROMPT = (
    'As an advanced reading comprehension assistant, your task is to analyze '
    'text passages and corresponding questions meticulously. '
    'Your response start after "Thought: ", where you will methodically break '
    'down the reasoning process, illustrating how you arrive at conclusions. '
    'Conclude with "Answer: " to present a concise, definitive response, '
    'devoid of additional elaborations.'
)

ONE_SHOT = (
    "Wikipedia Title: The Last Horse\n"
    "The Last Horse (Spanish: El último caballo) is a 1950 Spanish comedy film "
    "directed by Edgar Neville starring Fernando Fernán Gómez.\n"
    "Wikipedia Title: Southampton\n"
    "The University of Southampton, which was founded in 1862 and received its "
    "Royal Charter as a university in 1952, has over 22,000 students...\n"
    "Wikipedia Title: Neville A. Stanton\n"
    "Neville A. Stanton is a British Professor of Human Factors and Ergonomics "
    "at the University of Southampton...\n"
    "Question: When was Neville A. Stanton's employer founded?\n"
    "Thought: The employer of Neville A. Stanton is University of Southampton. "
    "The University of Southampton was founded in 1862.\n"
    "Answer: 1862.\n"
)

print('Metriche e prompts OK')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 14 – Checkpoint helpers (scrivono in RESULTS_FOLDER che cambia per run)
# ══════════════════════════════════════════════════════════════════════════════

def _iter_checkpoint_path() -> str:
    return os.path.join(RESULTS_FOLDER, 'checkpoint.json')

def _iter_final_path() -> str:
    return os.path.join(RESULTS_FOLDER, 'benchmark_result.json')

def _iter_save_partial(results_list: list) -> None:
    os.makedirs(RESULTS_FOLDER, exist_ok=True)
    with open(_iter_checkpoint_path(), 'w', encoding='utf-8') as f:
        json.dump({'results': results_list}, f, indent=2, ensure_ascii=False)

def _iter_load_checkpoint() -> Optional[list]:
    path = _iter_checkpoint_path()
    if os.path.isfile(path):
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        results = d.get('results', [])
        print(f'[RESUME] Checkpoint trovato: {len(results)} risultati già salvati.')
        return results
    return None


print('Checkpoint helpers OK')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 15 – run_iterative_benchmark
# ══════════════════════════════════════════════════════════════════════════════

async def run_iterative_benchmark(
    dataset,
    retriever_instance,
    k_docs           = K_DOCS,
    dataset_name     = DATASET_NAME,
    results_list     = None,
    checkpoint_every = CHECKPOINT_EVERY,
    extractor        = None,
    chunk_ents       = None,
):
    """
    Benchmark con recupero iterativo entity-aware.

    Per ogni domanda:
      1. iterative_retrieve → k_docs chunk
      2. Taglia al MAX_CTX_CHUNKS per il contesto LLM
      3. Chiama LLM → calcola EM / F1 / Recall@K (title-based)
      4. Checkpoint ogni checkpoint_every campioni in RESULTS_FOLDER
    """
    _extractor  = extractor  if extractor  is not None else globals().get('entity_extractor')
    _chunk_ents = chunk_ents if chunk_ents is not None else globals().get('chunk_entities', {})

    os.makedirs(RESULTS_FOLDER, exist_ok=True)

    if results_list is None:
        checkpoint   = _iter_load_checkpoint()
        results_list = checkpoint if checkpoint is not None else []

    already_done = len(results_list)
    dataset_todo = dataset[already_done:]

    if already_done > 0:
        print(f'[RESUME] Riprendo da item {already_done}/{len(dataset)}')
    print(f'--- Entity-Aware Iterative Benchmark: {len(dataset_todo)} item rimanenti ---')
    print(f'--- k_docs={k_docs}  MAX_CTX={MAX_CTX_CHUNKS}  Output: {RESULTS_FOLDER} ---')

    t0 = time.perf_counter()
    sum_ret_ms = sum_llm_ms = sum_tot_ms = 0.0
    count_valid = 0

    for i, item in enumerate(dataset_todo):
        global_i    = already_done + i
        question    = item['question']
        gold_answer = item['answer']

        gold_titles, gold_docs = [], []
        r_docs, iter_timings   = [], []
        total_ret_ms           = 0.0

        # ── Gold docs ──────────────────────────────────────────────────────────
        if 'wiki' in dataset_name.lower() or 'hotpot' in dataset_name.lower():
            needed = {f[0] for f in item.get('supporting_facts', [])}
            for title, sents in item.get('context', []):
                if title in needed:
                    gold_titles.append(title)
                    txt = ' '.join(sents)
                    gold_docs.append({'id': generate_doc_id(title, txt),
                                      'title': title, 'text': txt})
        elif 'musique' in dataset_name.lower():
            for p in item.get('paragraphs', []):
                if p.get('is_supporting'):
                    gold_titles.append(p['title'])
                    gold_docs.append({'id': generate_doc_id(p['title'], p['paragraph_text']),
                                      'title': p['title'], 'text': p['paragraph_text']})

        # ── Iterative retrieve ─────────────────────────────────────────────────
        t_query = time.perf_counter()
        try:
            ret_result   = await retriever_instance.iterative_retrieve(
                question, k=k_docs, extractor=_extractor, chunk_ents=_chunk_ents
            )
            r_docs        = ret_result['docs']
            iter_timings  = ret_result['timings']
            total_ret_ms  = ret_result['total_ms']
        except Exception as e:
            print(f'[ERR] iterative_retrieve(): {e}')

        pruned_docs = r_docs[:MAX_CTX_CHUNKS]
        ctx_text    = '\n\n'.join(d['text'].strip() for d in pruned_docs)

        # ── Recall@K title-based ───────────────────────────────────────────────
        retrieved_titles = [d["title"] for d in r_docs]

        # Deduplica mantenendo l'ordine di prima apparizione
        seen_t = set()
        unique_titles = []
        for t in retrieved_titles:
            if t not in seen_t:
                seen_t.add(t)
                unique_titles.append(t)

        if gold_titles:
            recall = {}
            for kk in [1, 2, 5, 10]:
                seent = set(unique_titles[:kk])   # <-- unica riga che cambia la logica
                recall[f"R{kk}"] = sum(1 for g in gold_titles if g in seent) / len(gold_titles)
        else:
            recall = {"R1": 0.0, "R2": 0.0, "R5": 0.0, "R10": 0.0}
    
    
        # ── LLM ───────────────────────────────────────────────────────────────
        t_llm = time.perf_counter()
        user_prompt = f'{ONE_SHOT}\nContext:\n{ctx_text}\nQuestion: {question}\nThought:'
        try:
            raw = await call_llm(user_prompt, system_prompt=SYSTEM_PROMPT)
            count_valid += 1
        except Exception as e:
            print(f'[ERR] call_llm(): {e}')
            raw = 'Answer: Error in generation.'
        llm_ms         = (time.perf_counter() - t_llm)   * 1000
        total_query_ms = (time.perf_counter() - t_query) * 1000

        sum_ret_ms += total_ret_ms
        sum_llm_ms += llm_ms
        sum_tot_ms += total_query_ms

        answer = extract_answer(raw)
        em     = compute_em(answer, gold_answer)
        f1     = compute_f1(answer, gold_answer)

        results_list.append({
            'id'              : item.get('id', f'idx{global_i}'),
            'question'        : question,
            'gold_answer'     : gold_answer,
            'gold_docs'       : gold_docs,
            'raw_response'    : raw,
            'generated_answer': answer,
            'em'              : em,
            'f1'              : f1,
            'num_retrieved'   : len(r_docs),
            'recall_at_k'     : recall,
            'retrieved_docs'  : r_docs,
            'context_len'     : len(ctx_text),
            'context'         : ctx_text,
            'timing_ms': {
                'total_retrieve_ms': round(total_ret_ms,   2),
                'llm_ms'           : round(llm_ms,         2),
                'total_ms'         : round(total_query_ms, 2),
                'per_iter'         : iter_timings,
            },
        })

        done_so_far = already_done + i + 1
        if done_so_far % checkpoint_every == 0:
            _iter_save_partial(results_list)
            print(f'  [checkpoint] {done_so_far}/{len(dataset)} → {_iter_checkpoint_path()}')

        if (i + 1) % 10 == 0:
            print(f'  [{done_so_far}/{len(dataset)}] '
                  f'EM={em} F1={f1:.3f} | '
                  f'retrieve={total_ret_ms:.0f}ms llm={llm_ms:.0f}ms')

    # ── Aggregazione finale ────────────────────────────────────────────────────
    total = len(results_list)
    if total == 0:
        print('Nessun risultato.')
        return None
    n_new = total - already_done

    iter_agg = defaultdict(lambda: {
        'embed_ms': [], 'search_ms': [], 'step_ms': [],
        'context_tokens': [], 'n_entities_used': [],
    })
    for r in results_list:
        for it in r['timing_ms']['per_iter']:
            s = it['iteration']
            iter_agg[s]['embed_ms'].append(it['embed_ms'])
            iter_agg[s]['search_ms'].append(it['search_ms'])
            iter_agg[s]['step_ms'].append(it['step_ms'])
            iter_agg[s]['context_tokens'].append(it['context_tokens'])
            iter_agg[s]['n_entities_used'].append(it['n_entities_used'])

    avg_per_iter = {
        s: {
            'avg_embed_ms'        : round(sum(v['embed_ms'])        / len(v['embed_ms']),        2),
            'avg_search_ms'       : round(sum(v['search_ms'])       / len(v['search_ms']),       2),
            'avg_step_ms'         : round(sum(v['step_ms'])         / len(v['step_ms']),         2),
            'avg_context_tokens'  : round(sum(v['context_tokens'])  / len(v['context_tokens']),  1),
            'avg_n_entities_used' : round(sum(v['n_entities_used']) / len(v['n_entities_used']), 2),
            'n_queries'           : len(v['embed_ms']),
        }
        for s, v in sorted(iter_agg.items())
    }

    summary = {
        'mean_em'          : sum(r['em'] for r in results_list) / total,
        'mean_f1'          : sum(r['f1'] for r in results_list) / total,
        'retrieval_recall' : {
            f'R@{kk}': sum(r['recall_at_k'].get(f'R@{kk}', 0) for r in results_list) / total
            for kk in [1, 2, 5, 10]
        },
        'total_samples'    : total,
        'k_docs'           : k_docs,
        'max_ctx_chunks'   : MAX_CTX_CHUNKS,
        'alpha'         : ALPHA,
        'sim_mode': SIM_MODE,
        'total_wall_time_s': round(time.perf_counter() - t0, 2),
        'index_type'       : INDEX_TYPE,
        'reranker'         : RERANKER,
        'results_folder'   : RESULTS_FOLDER,
        'avg_timing_ms': {
            'retrieve': round(sum_ret_ms / max(n_new, 1),       2),
            'llm'     : round(sum_llm_ms / max(count_valid, 1), 2),
            'total'   : round(sum_tot_ms / max(n_new, 1),       2),
        },
        'avg_per_iter': avg_per_iter,
        # avg cluster per livello: popolato da run_ablation_analysis dopo il build
        'avg_clusters_per_level': {},
    }

    final_path = _iter_final_path()
    with open(final_path, 'w', encoding='utf-8') as f:
        json.dump({'summary': summary, 'results': results_list}, f,
                  indent=4, ensure_ascii=False)

    cp = _iter_checkpoint_path()
    if os.path.isfile(cp):
        os.remove(cp)
        print(f'  Checkpoint rimosso: {cp}')

    print('\nBenchmark entity-aware iterativo completato!')
    print(f"  INDEX={summary['index_type']}  RERANKER={summary['reranker']}  K={k_docs}")
    print(f"  EM={summary['mean_em']:.4f}  F1={summary['mean_f1']:.4f}")
    print(f"  R@1={summary['retrieval_recall']['R@1']:.3f}  "
          f"R@5={summary['retrieval_recall']['R@5']:.3f}  "
          f"R@10={summary['retrieval_recall']['R@10']:.3f}")
    print(f"  Retrieve medio={summary['avg_timing_ms']['retrieve']}ms  "
          f"LLM medio={summary['avg_timing_ms']['llm']}ms")
    print(f'  Risultati salvati in: {final_path}')

    return {'summary': summary, 'results': results_list}


print('run_iterative_benchmark() OK')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 16 – Ablation helpers
# ══════════════════════════════════════════════════════════════════════════════

def _ablation_suffix(cfg: dict) -> str:
    alpha_str = str(cfg['IOU_RERANK']) if cfg['IOU_RERANK'] is not None else '0.0'
    sim = cfg.get('SIM_MODE', 'beta')
    return f"ctx{cfg['MAX_CTX_CHUNKS']}_cl{cfg['TOP_K_CLUSTERS_PER_DOC']}_alpha{alpha_str}_sim{sim}"



def _ablation_run_dir(suffix: str) -> str:
    """Directory dedicata alla singola run (checkpoint + risultato finale)."""
    return os.path.join(_ABLATION_ROOT, suffix)


def _ablation_result_path(suffix: str) -> str:
    return os.path.join(_ablation_run_dir(suffix), 'ablation_result.json')


def _ablation_done(suffix: str) -> bool:
    return os.path.isfile(_ablation_result_path(suffix))


def _ablation_apply(cfg: dict) -> None:
    """
    Sovrascrive i globali con i valori della configurazione corrente.

    Fissi: INDEX_TYPE='numpy'  RERANKER='biencoder'  K_DOCS=10
    Variabili: MAX_CTX_CHUNKS, TOP_K_CLUSTERS_PER_DOC, BETA_IOU, RESULTS_FOLDER

    RESULTS_FOLDER viene impostato alla directory della run corrente in modo
    che i checkpoint di run_iterative_benchmark (ogni CHECKPOINT_EVERY campioni)
    abbiano path univoci e non si sovrascrivano tra configurazioni diverse.
    """
    global INDEX_TYPE, RERANKER, K_DOCS
    global MAX_CTX_CHUNKS, TOP_K_CLUSTERS_PER_DOC
    global ALPHA, SIM_MODE
    global RESULTS_FOLDER

    INDEX_TYPE = 'numpy'
    RERANKER   = 'biencoder'
    K_DOCS     = 10

    MAX_CTX_CHUNKS         = cfg['MAX_CTX_CHUNKS']
    TOP_K_CLUSTERS_PER_DOC = cfg['TOP_K_CLUSTERS_PER_DOC']
    ALPHA = cfg['IOU_RERANK'] if cfg['IOU_RERANK'] is not None else 0.0
    SIM_MODE = cfg.get('SIM_MODE', 'beta')


    suffix         = _ablation_suffix(cfg)
    RESULTS_FOLDER = _ablation_run_dir(suffix)
    os.makedirs(RESULTS_FOLDER, exist_ok=True)


def _build_row(cfg: dict, suffix: str, smry: dict,
               avg_cl: dict = None) -> dict:
    """Costruisce la riga del CSV riepilogativo da config + summary."""
    row = {
        'config_suffix'          : suffix,
        'MAX_CTX_CHUNKS'         : cfg['MAX_CTX_CHUNKS'],
        'TOP_K_CLUSTERS_PER_DOC' : cfg['TOP_K_CLUSTERS_PER_DOC'],
        'IOU_RERANK'             : str(cfg['IOU_RERANK']),
        'ALPHA_used': cfg['IOU_RERANK'] if cfg['IOU_RERANK'] is not None else 0.0,
        'SIM_MODE'   : cfg.get('SIM_MODE', 'beta'),   # ← nuovo
        'mean_em'                : smry.get('mean_em',  float('nan')),
        'mean_f1'                : smry.get('mean_f1',  float('nan')),
        'R@1'  : smry.get('retrieval_recall', {}).get('R@1',  0),
        'R@2'  : smry.get('retrieval_recall', {}).get('R@2',  0),
        'R@5'  : smry.get('retrieval_recall', {}).get('R@5',  0),
        'R@10' : smry.get('retrieval_recall', {}).get('R@10', 0),
        'avg_retrieve_ms' : smry.get('avg_timing_ms', {}).get('retrieve', float('nan')),
        'avg_llm_ms'      : smry.get('avg_timing_ms', {}).get('llm',      float('nan')),
        'avg_total_ms'    : smry.get('avg_timing_ms', {}).get('total',    float('nan')),
        'total_samples'   : smry.get('total_samples', 0),
    }
    # avg cluster di appartenenza per livello, espansi come colonne
    _cl = avg_cl or smry.get('avg_clusters_per_level', {})
    for k, v in _cl.items():
        row[f'avg_cl_{k}'] = v
    return row


print('Ablation helpers OK')


# ══════════════════════════════════════════════════════════════════════════════
# CELLA 17 – run_ablation_analysis
# ══════════════════════════════════════════════════════════════════════════════

async def run_ablation_analysis(
    dataset,
    grid:         dict = None,
    n_samples:    int  = ABLATION_N_SAMPLES,
    dataset_name: str  = DATASET_NAME,
    extractor          = None,
    chunk_ents         = None,
):
    """
    Esegue l'ablation su tutte le combinazioni di _ABLATION_GRID.

    Per ogni configurazione:
      1. _ablation_apply(cfg)  → sovrascrive globali + RESULTS_FOLDER
      2. Carica da SAVE_DIR/topk_<N>/ se l'indice esiste,
         altrimenti lo costruisce e lo salva.
         Ogni valore di TOP_K_CLUSTERS_PER_DOC ha il suo indice separato
         (il clustering dipende da top_k_clusters_per_doc, quindi gli indici
         NON sono intercambiabili tra valori diversi).
      3. run_iterative_benchmark() su dataset[:n_samples]
         → checkpoint ogni CHECKPOINT_EVERY campioni in RESULTS_FOLDER
         → salva avg_clusters_per_level nel summary
      4. Salva ablation_result.json nella directory della run.
      5. Aggiorna ablation_summary.csv in _ABLATION_ROOT.

    Resume-safe: le run con ablation_result.json già presente vengono saltate.

    Path output:
      Indici  : SAVE_DIR/topk_<N>/
      Run     : _ABLATION_ROOT/<suffix>/ablation_result.json
      Sommario: _ABLATION_ROOT/ablation_summary.csv
    """
    _grid       = grid       or _ABLATION_GRID
    _extractor  = extractor  or globals().get('entity_extractor')
    _chunk_ents = chunk_ents or globals().get('chunk_entities', {})

    keys         = list(_grid.keys())
    combos       = list(_itertools.product(*[_grid[k] for k in keys]))
    total_combos = len(combos)

    os.makedirs(_ABLATION_ROOT, exist_ok=True)
    print(len(dataset))
    data_subset   = dataset[:n_samples]
    ablation_rows = []

    print(f"\n{'='*65}")
    print(f"  ABLATION — {total_combos} configurazioni × {n_samples} campioni")
    print(f"  Fissi: INDEX_TYPE=numpy  RERANKER=biencoder  K_DOCS=10")
    print(f"  Indici: {SAVE_DIR}/topk_<N>/")
    print(f"  Output: {_ABLATION_ROOT}")
    print(f"{'='*65}\n")

    for run_idx, combo in enumerate(combos, 1):
        cfg    = dict(zip(keys, combo))
        suffix = _ablation_suffix(cfg)

        print(f"[{run_idx:3d}/{total_combos}] {suffix}")

        # ── Resume ────────────────────────────────────────────────────────────
        if _ablation_done(suffix):
            print(f"           → già completata, carico i risultati.")
            with open(_ablation_result_path(suffix), 'r', encoding='utf-8') as fh:
                saved = json.load(fh)
            ablation_rows.append(_build_row(cfg, suffix, saved.get('summary', {})))
            continue

        # ── 1. Sovrascrive globali e RESULTS_FOLDER ───────────────────────────
        _ablation_apply(cfg)

        # ── 2. Carica o costruisce l'indice per TOP_K_CLUSTERS_PER_DOC ────────
        # IMPORTANTE: top_k_clusters_per_doc cambia il numero di cluster
        # assegnati a ogni documento durante _build_level → ogni valore
        # produce doc_clusters diversi → l'indice NON può essere riutilizzato
        # tra valori diversi.
        topk      = cfg['TOP_K_CLUSTERS_PER_DOC']
        topk_path = retriever_dir_for_topk(topk)

        if index_exists_for_topk(topk):
            print(f"           [indice] carico  topk={topk} ← {topk_path}")
            _retriever = load_retriever(topk_path)
        else:
            print(f"           [indice] costruisco topk={topk} → {topk_path}")
            _retriever = MatryoshkaDendrogramRetriever(
                top_k_clusters_per_doc=topk
            )
            await _retriever.build_kg(documents)
            save_retriever(_retriever, topk_path)
            print(f"           [indice] salvato in {topk_path}")

        # Media cluster per livello (salvata nel summary e nel CSV)
        avg_cl_per_level = _retriever.avg_clusters_per_level()
        avg_cl_leaf = avg_cl_per_level.get(
            f'L{len(_retriever.levels)-1}_dim{_retriever.dims[-1]}', float('nan')
        )
        print(f"           MAX_CTX={MAX_CTX_CHUNKS}  TOP_K={topk}  "
              f"ALPHA={ALPHA}  avg_cl_leaf={avg_cl_leaf:.2f}")

        # ── 3. Benchmark ──────────────────────────────────────────────────────
        try:
            result = await run_iterative_benchmark(
                dataset          = data_subset,
                retriever_instance = _retriever,
                k_docs           = K_DOCS,
                dataset_name     = dataset_name,
                results_list     = None,
                checkpoint_every = CHECKPOINT_EVERY,
                extractor        = _extractor,
                chunk_ents       = _chunk_ents,
            )
        except Exception as exc:
            import traceback
            print(f"           [ERRORE] {exc}")
            traceback.print_exc()
            result = None

        if result is None:
            print(f"           → run fallita, skip.")
            continue

        smry = result['summary']
        smry['avg_clusters_per_level'] = avg_cl_per_level   # aggiungi al summary

        # ── 4. Salva risultato della run ──────────────────────────────────────
        out = {
            'config'  : {k: str(v) for k, v in cfg.items()},
            'suffix'  : suffix,
            'summary' : smry,
            'results' : result['results'],
        }
        with open(_ablation_result_path(suffix), 'w', encoding='utf-8') as fh:
            json.dump(out, fh, indent=4, ensure_ascii=False)

        print(f"           EM={smry['mean_em']:.4f}  "
              f"F1={smry['mean_f1']:.4f}  "
              f"R@5={smry['retrieval_recall'].get('R@5', 0):.3f}  "
              f"R@10={smry['retrieval_recall'].get('R@10', 0):.3f}")

        ablation_rows.append(_build_row(cfg, suffix, smry, avg_cl_per_level))

        # ── 5. Aggiorna CSV intermedio dopo ogni run ───────────────────────────
        _csv_path = os.path.join(_ABLATION_ROOT, 'ablation_summary.csv')
        pd.DataFrame(ablation_rows).sort_values(
            'mean_f1', ascending=False
        ).reset_index(drop=True).to_csv(_csv_path, index=False, encoding='utf-8')

    # ── CSV riepilogativo finale ───────────────────────────────────────────────
    if not ablation_rows:
        print('[WARN] Nessuna riga raccolta.')
        return None

    df_abl   = (pd.DataFrame(ablation_rows)
                .sort_values('mean_f1', ascending=False)
                .reset_index(drop=True))
    csv_path = os.path.join(_ABLATION_ROOT, 'ablation_summary.csv')
    df_abl.to_csv(csv_path, index=False, encoding='utf-8')

    print(f"\n{'='*65}")
    print(f"  Ablation completata!  Riepilogo: {csv_path}")
    cols_show = ['config_suffix', 'MAX_CTX_CHUNKS', 'TOP_K_CLUSTERS_PER_DOC',
                 'IOU_RERANK', 'BETA_IOU_used', 'mean_em', 'mean_f1', 'R@5', 'R@10']
    print(df_abl[[c for c in cols_show if c in df_abl.columns]].head(8).to_string(index=False))
    print(f"{'='*65}\n")
    return df_abl

print('run_ablation_analysis() definita.')

# Run ablation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 18 – Avvio ablation
# ══════════════════════════════════════════════════════════════════════════════

ablation_df = await run_ablation_analysis(
    dataset      = data,
    n_samples    = ABLATION_N_SAMPLES,
    dataset_name = DATASET_NAME,
    extractor    = entity_extractor,
    chunk_ents   = chunk_entities,
)